In [ ]:
# ===== BASELINE GREEDY — CONFIG (edit ONLY this cell) =====================

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH   = "test/stable-ac-moves-w4"   # branch that contains experiments/
REPO_DIR = "ACSolverX"
CLONE       = True
UPDATE_REPO = True           # git reset --hard so a RESTART pulls latest push
MOUNT_DRIVE = True           # results -> Drive if True, else local ./results/

# --- experiment knobs ---------------------------------------------------
BUDGET = [1000, 5000, 10000, 25000, 50000]   # node budget per presentation; one jsonl per budget

cfg = {
    "DATASET": "data/ms640_solved.txt",
    "SUBSET": None,                 # None=all 640, list[int], or (start, end)
    "MAX_RELATOR_LENGTH": 24,       # per-relator cap (24 = ms640 layout)
    "CYCLIC_REDUCE": True,          # toggle cyclic reduction after each move

    # jsonl field toggles
    "use_min_relator_length": True,
    "use_min_relator": True,
    "use_max_relator_length": True,
    "use_max_relator": True,
    "use_time": True,
    "use_path": True,
    "PATH_IN_SEPARATE_FILE": True,  # solved paths -> *_paths.jsonl

    "RESUME": True,                 # skip pres_ids already in the jsonl

    "PROGRESS_EVERY": 10,           # print a status line every N presentations

    # output
    "MOUNT_DRIVE": MOUNT_DRIVE,
    "DRIVE_OUT_DIR": "/content/drive/MyDrive/acsolverx_results/greedy_baseline",
    "LOCAL_OUT_DIR": "results/greedy_baseline",

    # Weights & Biases
    "USE_WANDB": True,
    "WANDB_ENTITY": "avigyapaudel045-aisc",   # writable team entity (org-managed acct); None = account default
    "WANDB_PROJECT": "acsolver",
    "WANDB_MODE": "online",         # "offline" for flaky Colab -> wandb sync later
    "WANDB_GROUP": None,            # default: greedy_baseline_{date}
}

print("config loaded.")

In [ ]:
# ==================== SETUP (clone / pull / install / mount) ==============
import os, sys, subprocess

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)                       # anchor so re-runs never nest the clone
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy wandb")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    # local: walk up from cwd to the repo root (dir holding experiments/ + data/)
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

# run from repo root so "data/..." and "import experiments..." resolve
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)

# W&B auth: prefer a Colab Secret named WANDB_API_KEY (add it via the key
# icon in Colab's left sidebar and toggle notebook access ON). If it's not
# found, fall back to the interactive wandb.login() prompt.
if cfg["USE_WANDB"]:
    import wandb
    if not os.environ.get("WANDB_API_KEY"):
        _key = None
        try:
            from google.colab import userdata
            _key = userdata.get("WANDB_API_KEY")
        except Exception:
            _key = None
        if _key:
            os.environ["WANDB_API_KEY"] = _key
            print("W&B: using Colab Secret WANDB_API_KEY (no prompt).")
        else:
            import getpass
            print("W&B: no WANDB_API_KEY secret found.")
            os.environ["WANDB_API_KEY"] = getpass.getpass(
                "Paste your W&B API key (from wandb.ai/authorize), then Enter: "
            ).strip()
    # verify the key against the W&B server (real round-trip)
    try:
        wandb.login(key=os.environ["WANDB_API_KEY"], verify=True)
        try:
            _default = wandb.Api().default_entity
        except Exception:
            _default = None
        # Runs are PINNED to cfg's entity/project via wandb.init(entity=...),
        # regardless of the account's default entity.
        _target = cfg["WANDB_ENTITY"] or _default
        print(f"W&B: authenticated \u2713  runs -> {_target}/{cfg['WANDB_PROJECT']}")
        if cfg["WANDB_ENTITY"] and _default and _default != cfg["WANDB_ENTITY"]:
            print(f"     note: account default entity is '{_default}', but these "
                  f"runs are pinned to '{cfg['WANDB_ENTITY']}' (edit WANDB_ENTITY in CONFIG to change).")
        elif not cfg["WANDB_ENTITY"]:
            print(f"     (WANDB_ENTITY is None -> using account default '{_default}'. "
                  f"Set it to your username to pin a different entity.)")
    except Exception as e:
        print(f"W&B: authentication FAILED \u2717 -- {e}")

In [ ]:
# ==================== RUN =================================================
from experiments.run_baseline import run_dataset

for budget in BUDGET:
    run_dataset(cfg, node_budget=budget)   # resumable; one jsonl per budget